# Where do the speedups actually come from? — leave-one-out attribution on Kaggle T4

Run top-to-bottom in a fresh **GPU T4 x2** session (only GPU 0 is used; the second stays idle
so nothing competes for power or PCIe). Every command is fail-fast; do not continue past a
failed gate.

## Why this run exists

The optimization table in the README is **historical, not attributable**. Each row was measured
against whatever baseline existed the day it landed, so the numbers do not compose and several
of them overlap:

- "CUDA graphs on decode, −72%" was measured before chunked prefill and the fused step existed.
- The fused step partly subsumes prefill graphs — −56.7% and −17.6% are not independent.
- "Warmup, −71% p99" only has anything to remove because graph capture exists at all.

So *"where does 1,069 tok/s come from?"* is unanswered. This notebook answers it with
**leave-one-out**: build the full recommended configuration, then remove one optimization at a
time, interleaved `full / minus_a / full / minus_b / ...` so thermal drift lands on every arm
equally, with the stock-Transformers token gate active on every engine.

**Read the sign carefully.** Every comparison is `minus_X` **vs** `full`. A *positive* change in
a latency metric means removing X made things **worse**, so X was contributing that much. The
column is the **cost of removal**, not a speedup.

Expect the leave-one-out numbers to be **smaller** than the historical table, and expect some to
come back `unresolved`. That is the finding, not a failure: at a given operating point, only
some of these are load-bearing.

## What this covers, and what it cannot

Seven optimizations have toggles and are measured directly: CUDA graphs, prefix cache, prefill
graphs, fused step, and the three Triton elementwise fusions. Two more are measured as separate
two-arm settings in section 5: warmup and the SDPA prefill kernel.

**Not measurable by ablation:** continuous batching, the paged KV cache, preemption, and
slice-copy metadata staging have no toggle — removing them is not a configuration, it is a
different engine. Those belong to the commit ladder in `docs/t4-reevaluation-plan.md`. Report
seven-plus-two and say so, rather than implying the table covers all twelve.

## Shape of the run

Four operating points, because the whole point is that attribution moves with the workload:

| cell | profile | concurrency | why |
|---|---|---|---|
| chat_c8 | chat (~288–1024 tok) | 8 | the published headline configuration |
| long_c8 | long (~1088–2560 tok) | 8 | prefill-heavy — where fused step and prefill graphs should matter most |
| chat_c2 | chat | 2 | low concurrency — batching amortizes least, launch overhead matters most |
| chat_c16 | chat | 16 | high concurrency — where per-step overhead should wash out |

`chat_c8` runs 5 repeats; the rest run 3, which your d₂ control-chart constants support.
Budget roughly **3–4 hours** total. Kaggle allows 9 h per session.

In [ ]:
import json, os, re, shlex, subprocess
from datetime import datetime, timezone
from pathlib import Path
RUN = Path('results/kaggle-attribution-' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ'))
RUN.mkdir(parents=True, exist_ok=False)
os.environ['RUN'] = str(RUN.resolve())
def run(command):
    print('+', command)
    subprocess.run(command, shell=True, check=True, env=os.environ.copy())
print(RUN)

In [ ]:
run('bash scripts/setup_kaggle.sh')
run('python scripts/kaggle_preflight.py --out "$RUN/preflight.json"')
run('git rev-parse HEAD > "$RUN/git_sha.txt"')
run('git status --short > "$RUN/git_status.txt"')
run('nvidia-smi --query-gpu=name,driver_version,memory.total,clocks.max.sm --format=csv > "$RUN/gpu.txt"')
print(open(RUN / 'gpu.txt').read())

In [ ]:
# Gate: a wrong engine must not produce an attribution table.
run('python -m pytest -q -m "not cuda" -p no:cacheprovider')
run('python -m pytest -q -m cuda -p no:cacheprovider')

## 4. The leave-one-out matrix

Each invocation builds eight engines — `full` plus seven with one optimization removed — and
runs them interleaved for 30 s each, `--repeats` times. Every engine is gated against stock
Transformers for the first 8 greedy tokens before any timing happens; `loo_*` settings are
exempt from the *cross-arm* identity check (removing a Triton kernel legitimately changes the
reduction order) but **not** from the stock gate, which is what catches a wrong kernel.

Do not pass `--allow-token-drift`. If an arm trips the stock gate, that is the result.

In [ ]:
MATRIX = [
    # label      profile  concurrency  max_active  num_blocks  repeats
    ('chat_c8',  'chat',   8,           8,          1024,       5),
    ('long_c8',  'long',   8,           8,          2048,       3),
    ('chat_c2',  'chat',   2,           2,          512,        3),
    ('chat_c16', 'chat',   16,          16,         2048,       3),
]
for label, profile, concurrency, max_active, num_blocks, repeats in MATRIX:
    out = f'$RUN/loo_{label}.json'
    print(f'\n{"=" * 70}\n  leave-one-out: {label}  ({profile}, concurrency {concurrency}, '
          f'{num_blocks * 16} KV tokens, {repeats} repeats)\n{"=" * 70}')
    run('python -m benchmarks.reliability.ab'
        ' --setting loo_all'
        ' --model Qwen/Qwen3-0.6B --dtype float16'
        f' --prompt-profile {profile}'
        f' --concurrency {concurrency} --max-active {max_active}'
        f' --num-blocks {num_blocks}'
        f' --repeats {repeats} --duration 30'
        f' --out "{out}"')

## 5. The two that leave-one-out cannot reach

`warmup` is not an engine kwarg and the SDPA prefill kernel has no "off" — each is a separate
two-arm A/B. Both are short.

In [ ]:
EXTRA = [
    # setting              profile  concurrency  max_active  num_blocks  repeats  why
    ('warmup',             'chat',   8,          8,          1024,       3),   # optimization 10
    ('prefill_sdpa',       'long',   8,          8,          2048,       3),   # optimization 5
    ('prefill_chunk_large','long',   8,          8,          2048,       3),   # informs 4
]
for setting, profile, concurrency, max_active, num_blocks, repeats in EXTRA:
    print(f'\n{"=" * 70}\n  {setting}  ({profile}, concurrency {concurrency})\n{"=" * 70}')
    run('python -m benchmarks.reliability.ab'
        f' --setting {setting}'
        ' --model Qwen/Qwen3-0.6B --dtype float16'
        f' --prompt-profile {profile}'
        f' --concurrency {concurrency} --max-active {max_active}'
        f' --num-blocks {num_blocks}'
        f' --repeats {repeats} --duration 30'
        f' --out "$RUN/extra_{setting}.json"')

## 6. The attribution table

`cost of removal` is `minus_X` measured against `full`. **Positive means the optimization was
earning its place**: without it, that metric got worse by this much. `unresolved` means the
change sat inside the run-to-run spread of both arms — it is a real answer, not a missing one.

In [ ]:
import pandas as pd

VERDICT = re.compile(r'^(?P<unres>unresolved: )?(?P<change>[-+][\d.]+)% .*?(?P<spread>[\d.]+)%')

def parse(verdict: str):
    # -> (change %, spread %, resolved) from ab.py's verdict string, or Nones
    if not isinstance(verdict, str):
        return None, None, None
    m = VERDICT.match(verdict)
    if not m:
        return None, None, None          # 'no data' / 'baseline is zero'
    return float(m['change']), float(m['spread']), m['unres'] is None

HEADLINE = ['step_timing.expected_gap_ms', 'latency.itl_p50', 'latency.itl_p99',
            'latency.ttft_p50', 'step_timing.decode_step_p50_ms',
            'step_timing.prefill_step_p50_ms']

rows, identity, gaps = [], [], []
for path in sorted(RUN.glob('loo_*.json')):
    payload = json.loads(path.read_text())
    cell = path.stem.replace('loo_', '')
    identity.append({'cell': cell, 'setting': payload['setting'],
                     'repeats': payload['repeats'],
                     'arms_identical': payload['token_identity']['identical'],
                     'failed_stock_gate': payload['token_identity']['early_divergence_arms'],
                     'gate_overridden': payload['token_identity']['gate_overridden']})
    # Read lazy captures as an ABSOLUTE per-arm value, never as a comparison verdict:
    # when the baseline is zero `_verdict` returns 'baseline is zero' whatever the variant
    # did, so a real regression would be invisible in `comparisons`.
    for arm_label, arm in payload.get('arms', {}).items():
        lazy = arm.get('summary', {}).get('step_timing.lazy_graph_captures', {})
        if lazy.get('max', 0):
            gaps.append({'cell': cell, 'arm': arm_label, 'max_lazy_captures': lazy['max'],
                         'median': lazy.get('median')})
        if not arm.get('ok', True):
            gaps.append({'cell': cell, 'arm': arm_label, 'max_lazy_captures': '-',
                         'median': f"INVARIANT VIOLATIONS: {arm.get('violations')}"})
    for arm, metrics in payload['comparisons'].items():
        row = {'cell': cell, 'removed': arm.replace('minus_', '')}
        for metric in HEADLINE:
            change, spread, resolved = parse(metrics.get(metric))
            short = metric.split('.')[-1].replace('_ms', '').replace('_p50', ' p50')
            row[short] = (f'{change:+.1f}%' if resolved else
                          (f'({change:+.1f}%)' if change is not None else '—'))
        rows.append(row)

loo = pd.DataFrame(rows)
print('=== cost of removing each optimization  (positive = it was earning its place) ===')
print('    (bracketed values are unresolved: inside run-to-run spread)\n')
for cell, df in loo.groupby('cell'):
    print(f'--- {cell} ---')
    display(df.drop(columns='cell').set_index('removed'))

print('\n=== token-identity gate, per cell ===')
display(pd.DataFrame(identity))
if gaps:
    print('\n!!! arms that captured a graph inside a timed window, or violated an invariant.')
    print('    Their tail percentiles are NOT results:')
    display(pd.DataFrame(gaps))
else:
    print('\nlazy graph captures: zero in every arm; no invariant violations. Tails are usable.')

print('\n=== two-arm settings ===')
for path in sorted(RUN.glob('extra_*.json')):
    payload = json.loads(path.read_text())
    print(f"\n--- {payload['setting']}  ({payload['prompt_profile']}) ---")
    for metric in HEADLINE:
        print(f"  {metric:34s} {payload['comparison'].get(metric)}")

loo.to_csv(RUN / 'attribution.csv', index=False)
print(f'\nwrote {RUN / "attribution.csv"}')

## 7. What to paste back

The CSV plus the two-arm block above is everything needed to replace the README's historical
table with an attributable one. Three things are worth reading before you do:

1. **Which optimizations are load-bearing at which operating point.** If `graphs` dominates at
   concurrency 2 and fades at 16, that is the per-step-overhead story your vLLM comparison
   already showed, now measured from the inside.
2. **Which come back `unresolved` everywhere.** Those are honest nulls. Say so.
3. **Whether any arm failed the stock-Transformers gate.** If one did, that optimization is
   changing output, and no speedup it shows is admissible.

Then zip the run directory off Kaggle:

```python
import shutil; shutil.make_archive('/kaggle/working/attribution', 'zip', RUN)
```